# 06 lora frame

Question: does the LoRa frame of `aosabi-carrier-lora` seal, open and refuse as the Homa design
note, second version, section 6.5 states, and does no nonce repeat across restarts.

The side under test is `aosabi-devtool lora` over the crate `aosabi-carrier-lora`: its seal and its
open of a frame, the AEAD of `chacha20poly1305` the seal takes, the carrier's receive path that
hands Homa a packet or refuses the frame, and its counters across restarts. The reference side is
`tests/harness/homa/lora.py` over `tests/harness/homa/chacha.py`, ChaCha20-Poly1305 of RFC 8439
written in the standard library and sharing no code with the crates, held first to the two vectors
of AEAD_CHACHA20_POLY1305 that RFC 8439 publishes in its section 2.8.2 and its appendix A.5, which
`tests/harness/homa/vectors.py` parsed out of the pinned text into
`tests/capture/homa_vectors/chacha20poly1305.json`.

A frame is a byte of the link, the sender's counter in four bytes big endian, the compact Homa
packet sealed and the tag of 16 bytes over the header and the sealed bytes, and its nonce is the
link, the byte of the direction, six zero bytes and the counter.

The exit number, stage H6 of the plan's section The notebooks: 0 differences on the vectors, 100
percent of altered frames refused before Homa, 0 nonces reused across 100 simulated restarts, 0
panics.

Judged stage: H6 of the Homa track.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The seed of the drawn keys, links, counters, packets and alterations.
seed = 20260928

# The seeded frames sealed and opened on both sides.
frames_ops = 1000

# The restarts of the counters and of the carrier.
restarts = 100

# The per command timeout, and the one of a cold build.
timeout_s = 600
build_timeout_s = 900

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain, the interface line of `lora`, and the gates of the plan's section
The commands the notebook rests on: the build of the devtool, the lints of the carrier with every
feature, its build for `riscv32imc-unknown-none-elf`, the RV32IMC core of the HT-CT62, with and
without the driver of the SX1262, and its format under the edition 2024 of the workspace, every
cargo call with `--locked`.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run
from harness.homa import chacha, lora as ref

P = paths(aosabi_root)
OUT = P.out / "06_lora_frame"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTFMT = CARGO_BIN / "rustfmt"
DEVICE = "riscv32imc-unknown-none-elf"

GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked -p aosabi-carrier-lora --all-features -- -D warnings",
     [CARGO, "clippy", "--locked", "-p", "aosabi-carrier-lora", "--all-features", "--", "-D", "warnings"]),
    ("cargo build --locked -p aosabi-carrier-lora --release, the device",
     [CARGO, "build", "--locked", "-p", "aosabi-carrier-lora", "--release", "--target", DEVICE]),
    ("cargo build --locked -p aosabi-carrier-lora --release, the device with sx1262",
     [CARGO, "build", "--locked", "-p", "aosabi-carrier-lora", "--release", "--target", DEVICE,
      "--features", "sx1262"]),
    ("rustfmt --check --edition 2024 crates/aosabi-carrier-lora/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-carrier-lora" / "src" / "lib.rs"]),
]
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=build_timeout_s, check=False)
    print("gate %-78s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "lora", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("cargo          :", reported_cargo)
print("seam           :", json.dumps(smoke))
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, reported_cargo
assert smoke == {"subcommand": "lora", "implemented": True, "lands_at": "H6"}, smoke

PANICS = []
ROWS = []


def row(name, value, bound, ok, mark="MEASURED"):
    """Record one row of the exit number with its measure, its bound and its state."""
    state = "ok" if ok else "CROSSED"
    ROWS.append((name, value, bound, state, mark))
    print("%-62s %-14s %-14s %s" % (name[:62], value, bound, state))


def lora(requests, name):
    """Send requests to `lora` in one process and return its answers."""
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run([DEVTOOL, "lora", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False)
    answers = [json.loads(line) for line in result.out.splitlines()]
    if result.returncode != 0 or len(answers) != len(requests):
        PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
            name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
    bad = [a for a in answers if "fatal" in a]
    assert not bad, "the seam refused requests of %s: %s" % (name, bad[:2])
    return answers


def pairing(key, link):
    return {"link": link, "key": key.hex()}

## The vectors of RFC 8439

The two vectors of AEAD_CHACHA20_POLY1305 of RFC 8439, section 2.8.2, which encrypts, and appendix
A.5, which decrypts, each with its key, its nonce of 12 bytes, its associated data, its plaintext,
its ciphertext and its tag. The cell holds the reference to both vectors first, then sends each to
the AEAD the frame seals with, `aead` and `aead_open` of the devtool, and counts every byte of the
ciphertext, the tag and the plaintext that differs from the published one. A tag with one bit
flipped must be refused on both sides.

In [ ]:
VECTORS = json.loads((P.tests / "capture" / "homa_vectors" / "chacha20poly1305.json").read_text(encoding="utf-8"))
print("source:", VECTORS["sources"][0]["url"], "sha256", VECTORS["sources"][0]["sha256"][:16], "read", VECTORS["sources"][0]["read"])
vector_differences = 0
requests = []
for index, vector in enumerate(VECTORS["vectors"]):
    b = {k: bytes.fromhex(vector[k]) for k in ("key", "nonce", "aad", "plaintext", "ciphertext", "tag")}
    sealed, tag = chacha.seal(b["key"], b["nonce"], b["aad"], b["plaintext"])
    opened = chacha.open_(b["key"], b["nonce"], b["aad"], b["ciphertext"], b["tag"])
    assert (sealed, tag, opened) == (b["ciphertext"], b["tag"], b["plaintext"]), "the reference misses %s" % vector["name"]
    flipped = bytes([b["tag"][0] ^ 1]) + b["tag"][1:]
    assert chacha.open_(b["key"], b["nonce"], b["aad"], b["ciphertext"], flipped) is None
    common = {"key": vector["key"], "nonce": vector["nonce"], "aad": vector["aad"]}
    requests.append(dict(common, id=3 * index, op="aead", hex=vector["plaintext"]))
    requests.append(dict(common, id=3 * index + 1, op="aead_open", hex=vector["ciphertext"], tag=vector["tag"]))
    requests.append(dict(common, id=3 * index + 2, op="aead_open", hex=vector["ciphertext"], tag=flipped.hex()))
answers = lora(requests, "vectors")
for index, vector in enumerate(VECTORS["vectors"]):
    sealed, opened, refused = answers[3 * index: 3 * index + 3]
    got = bytes.fromhex(sealed["hex"]) + bytes.fromhex(sealed["tag"]) + bytes.fromhex(opened["hex"])
    want = bytes.fromhex(vector["ciphertext"] + vector["tag"] + vector["plaintext"])
    differing = sum(1 for x, y in zip(got, want) if x != y) + abs(len(got) - len(want))
    vector_differences += differing
    print("%-26s %4d bytes of ciphertext, tag and plaintext, %d differ, a flipped tag answers %s" % (
        vector["name"], len(want), differing, refused.get("error")))
    if refused.get("error") != "Tag":
        vector_differences += 1
row("byte differences on the RFC 8439 vectors", vector_differences, "0", vector_differences == 0)

## 1,000 seeded frames

Each frame draws a key of 32 bytes, a link from 1 to 255, a direction, a counter below the spent
counter `0xFFFFFFFF` and a packet of 1 to 234 bytes, the longest a frame seals. The devtool seals
each packet and opens each frame on its own, and the reference does the same, and the cell counts
every byte and every answer that differs. A packet of 235 bytes and the spent counter are refused
on both sides.

In [ ]:
rng = random.Random(seed)
FRAMES = []
for i in range(frames_ops):
    key = bytes(rng.getrandbits(8) for _ in range(32))
    link = rng.randint(1, 255)
    direction = rng.choice([ref.TO_DEVICE, ref.TO_CLIENT])
    counter = rng.choice([0, 1, 0xFFFFFFFE, rng.randrange(0, 0xFFFFFFFF)]) if i % 50 == 0 else rng.randrange(0, 0xFFFFFFFF)
    packet = bytes(rng.getrandbits(8) for _ in range(rng.randint(1, ref.MAX_SEALED)))
    FRAMES.append((key, link, direction, counter, packet))
requests = [{"id": i, "op": "seal", "pairing": pairing(k, l), "direction": d, "counter": c, "hex": p.hex()}
            for i, (k, l, d, c, p) in enumerate(FRAMES)]
k0, l0, d0 = FRAMES[0][:3]
edge = [
    {"id": frames_ops, "op": "seal", "pairing": pairing(k0, l0), "direction": d0, "counter": ref.SPENT, "hex": "10"},
    {"id": frames_ops + 1, "op": "seal", "pairing": pairing(k0, l0), "direction": d0, "counter": 1, "hex": "10" * (ref.MAX_SEALED + 1)},
]
sealed = lora(requests + edge, "seal")
seal_differences = 0
SEALED = []
for (key, link, direction, counter, packet), answer in zip(FRAMES, sealed):
    want = ref.seal(key, link, direction, counter, packet)
    got = bytes.fromhex(answer["hex"]) if "hex" in answer else answer.get("error")
    if got != want:
        seal_differences += 1
    SEALED.append(want)
edge_answers = [a.get("error") for a in sealed[frames_ops:]]
edge_reference = [ref.seal(k0, l0, d0, ref.SPENT, b"\x10"), ref.seal(k0, l0, d0, 1, b"\x10" * (ref.MAX_SEALED + 1))]
seal_differences += sum(1 for a, b in zip(edge_answers, edge_reference) if a != b)
print("sealed:", len(FRAMES), "frames of", min(len(f) for f in SEALED), "to", max(len(f) for f in SEALED), "bytes,",
      seal_differences, "differ; the spent counter and 235 bytes answer", edge_answers, "and the reference", edge_reference)
requests = [{"id": i, "op": "open", "pairing": pairing(k, l), "direction": d, "frames": [SEALED[i].hex()]}
            for i, (k, l, d, c, p) in enumerate(FRAMES)]
opened = lora(requests, "open")
open_differences = 0
for (key, link, direction, counter, packet), frame, answer in zip(FRAMES, SEALED, opened):
    want = ref.open_all(key, link, direction, [frame])
    if answer["results"] != want or want[0].get("hex") != packet.hex():
        open_differences += 1
print("opened:", len(FRAMES), "frames,", open_differences, "differ from the reference")
row("byte differences of the 1,000 seeded frames, sealed and opened", seal_differences + open_differences, "0",
    seal_differences + open_differences == 0)

## Altered frames

Every seeded frame is altered eight ways: one bit flipped in the link, in the counter, in the sealed
packet and in the tag, the tag truncated by 1 to 16 bytes, a receiver of another link, the same
frame received twice, and a frame received after one whose counter lies 64 or more above it, below
the window. The devtool hands each altered frame to the carrier of the receiving end through
`receive`, whose answer names every packet the carrier handed Homa, so a frame refused there never
reached Homa. A genuine frame ahead of a replay or of a counter below the window must reach Homa
once. `open` answers the name of each refusal, which the reference must name as well.

In [ ]:
def flip(frame, lo, hi):
    """Return `frame` with one drawn bit of the bytes from `lo` to `hi` flipped."""
    at = rng.randrange(lo, hi)
    return frame[:at] + bytes([frame[at] ^ (1 << rng.randrange(8))]) + frame[at + 1:]


altered = []
for i, ((key, link, direction, counter, packet), frame) in enumerate(zip(FRAMES, SEALED)):
    other = link % 255 + 1
    cut = rng.randint(1, ref.TAG_LEN)
    later = min(counter + 64 + rng.randrange(0, 1000), 0xFFFFFFFE)
    below = counter if later - counter >= ref.WINDOW else None
    ahead = ref.seal(key, link, direction, later, packet)
    cases = [
        ("link", key, link, [flip(frame, 0, 1)], 0),
        ("counter", key, link, [flip(frame, 1, 5)], 0),
        ("sealed", key, link, [flip(frame, 5, len(frame) - ref.TAG_LEN)], 0),
        ("tag", key, link, [flip(frame, len(frame) - ref.TAG_LEN, len(frame))], 0),
        ("truncated tag", key, link, [frame[:-cut]], 0),
        ("another link", key, other, [frame], 0),
        ("replay", key, link, [frame, frame], 1),
    ]
    if below is not None:
        cases.append(("below the window", key, link, [ahead, frame], 1))
    for name, k, l, frames, genuine in cases:
        altered.append((i, name, k, l, direction, frames, genuine))

requests = [{"id": n, "op": "receive", "pairing": pairing(k, l), "direction": d, "frames": [f.hex() for f in frames]}
            for n, (i, name, k, l, d, frames, genuine) in enumerate(altered)]
received = lora(requests, "receive")
requests = [{"id": n, "op": "open", "pairing": pairing(k, l), "direction": d, "frames": [f.hex() for f in frames]}
            for n, (i, name, k, l, d, frames, genuine) in enumerate(altered)]
named = lora(requests, "refuse")
by_kind = {}
total = 0
refused = 0
naming = 0
for (i, name, k, l, d, frames, genuine), got, answer in zip(altered, received, named):
    handed = len(got["packets"])
    total += 1
    ok = handed == genuine
    refused += 1 if ok else 0
    want = ref.open_all(k, l, d, frames)
    if answer["results"] != want:
        naming += 1
    kind = by_kind.setdefault(name, [0, 0, set()])
    kind[0] += 1
    kind[1] += 1 if ok else 0
    kind[2].add(want[-1].get("error"))
for name, (count, ok, errors) in by_kind.items():
    print("%-18s %5d altered, %5d refused before Homa, refused as %s" % (name, count, ok, ", ".join(sorted(str(e) for e in errors))))
print("names of the refusals that differ from the reference:", naming)
row("altered frames refused before Homa, per cent", round(100.0 * refused / total, 3), "100", refused == total)
row("names of the refusals that differ from the reference", naming, "0", naming == 0)

## Counters across restarts

The counters of a link reserve blocks of 1,024 in the storage of the card and start after the last
reserved block at boot. `counters` boots one store 101 times, 100 restarts, taking a drawn count of
0 to 3,000 counters each time, and the reference computes the first counter and the store's bound of
each boot from the same rule. `restarts` restarts one carrier 100 times in each direction and sends
0 to 20 frames between restarts, and the cell reads the counter of every frame the carrier put on
the air. A nonce is the link, the direction and the counter, so a counter taken twice by one link
and direction is a nonce reused.

In [ ]:
uses = [rng.randint(0, 3000) for _ in range(restarts + 1)]
boots = lora([{"id": 0, "op": "counters", "link": 7, "direction": ref.TO_CLIENT, "uses": uses}], "counters")[0]["boots"]
model = ref.boots(uses)
boot_differences = sum(1 for got, want in zip(boots, model) if got != want) + abs(len(boots) - len(model))
taken = []
for boot in boots:
    taken.extend(range(boot["first"], boot["first"] + boot["count"]))
reused_counters = len(taken) - len(set(taken))
print("counters: %d boots, %d counters taken, the first of each boot from %d to %d, %d boots differ from the rule, %d taken twice" % (
    len(boots), len(taken), boots[0]["first"], boots[-1]["first"], boot_differences, reused_counters))

key = bytes(rng.getrandbits(8) for _ in range(32))
requests = []
for index, direction in enumerate([ref.TO_DEVICE, ref.TO_CLIENT]):
    sends = [rng.randint(0, 20) for _ in range(restarts + 1)]
    requests.append({"id": index, "op": "restarts", "pairing": pairing(key, 9), "direction": direction,
                     "sends": sends, "len": 30})
reused_frames = 0
frames_on_air = 0
for answer in lora(requests, "restarts"):
    counters = [f["counter"] for f in answer["frames"]]
    frames_on_air += len(counters)
    reused_frames += len(counters) - len(set(counters))
    by_boot = {}
    for f in answer["frames"]:
        by_boot.setdefault(f["boot"], []).append(f["counter"])
    order = [by_boot[b] for b in sorted(by_boot)]
    rising = all(max(a) < min(b) for a, b in zip(order, order[1:]))
    print("restarts %-9s %d frames on the air over %d boots with frames, counters of every boot above the last: %s" % (
        answer["direction"], len(counters), len(by_boot), rising))
    reused_frames += 0 if rising else 1
row("nonces reused across 100 simulated restarts", reused_counters + reused_frames, "0",
    reused_counters + reused_frames == 0 and boot_differences == 0)

## Verdict

Every row of the exit number prints its measure beside its bound, and a row that crosses its bound
prints CROSSED and fails the notebook. A call of the seam that exits other than 0 or answers fewer
lines than it was asked is a panic, and one fails the notebook.

In [ ]:
print("panics or crashes of the seam:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
print()
print("%-62s %-14s %-14s %s" % ("row", "measured", "bound", "state"))
for name, value, bound, state, mark in ROWS:
    print("%-62s %-14s %-14s %s" % (name[:62], value, bound, state))
crossed = [name for name, _, _, state, _ in ROWS if state != "ok"]
print()
print("VERDICT: %d of %d rows hold, %d panics%s" % (
    len(ROWS) - len(crossed), len(ROWS), len(PANICS), "" if not crossed else ". CROSSED: " + "; ".join(crossed)))
assert not PANICS, PANICS
assert not crossed, crossed